<a href="https://colab.research.google.com/github/Deepak-Bhardwaj/fde-training-day18/blob/main/GX_Core_FDE_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

# 🧪 Great Expectations (GX Core 1.0) — Hands-On Lab

### Data quality, made declarative — from the **Forward Deployed Engineer's** chair

`expect(your_data).to_be(trustworthy)`

**TECHADEMY · FDE TRAINING · DATA QUALITY**

</div>

---

### Why this lab is not a tutorial

A tutorial teaches you the API. This lab teaches you the **job**.

As an FDE you get dropped into a customer's environment where:

- there is no data dictionary,
- the person who built the pipeline left six months ago,
- everyone *says* the data is fine, and
- the dashboard has been quietly wrong for eleven days.

Great Expectations is not primarily a Python library in that situation. It is a **mechanism for extracting tribal knowledge out of people's heads and freezing it into an artifact that survives your departure.** The API is the easy part — you'll have it in twenty minutes. The judgment around it is the whole engagement.

---

### Learning outcomes

By the end of this notebook you will be able to:

| # | Outcome |
|---|---------|
| 1 | Explain data quality as six checkable dimensions, not a vibe |
| 2 | Run the full GX Core loop: Context → Source → Asset → Batch → Expectation → Validate |
| 3 | Turn profiling insights into an enforceable **Expectation Suite** (a data contract) |
| 4 | Wire a **Checkpoint** as a pass/fail gate with quarantine + alerting |
| 5 | Distinguish an expectation that **failed** from one that **errored** — and why FDEs care |
| 6 | Tier rules by severity so the gate doesn't die of alert fatigue |
| 7 | Produce a **handoff artifact** the customer's team can own after you leave |

---

### How to use this notebook

- **Runtime → Run all** works end-to-end, or step through with `Shift+Enter`.
- Cells marked **🎯 Your Turn** are yours to write. Solutions are at the bottom — resist.
- Cells marked **🧭 FDE Lens** are the point of the lab. Read them slowly.
- Everything runs on a free CPU runtime. No credentials, no cloud account, no customer data.

⏱️ Expected time: **75–90 minutes**

---
## 📋 The engagement brief

> **Customer:** MetroRide Mobility — a ride-hailing operator, ~40k trips/day.
> **You:** Forward Deployed Engineer, Day 3 of a four-week engagement.
>
> **What happened:** For eleven days the CFO's revenue dashboard under-reported daily
> gross bookings by roughly 8%. Nobody caught it. Finance found it by hand, reconciling
> against the payment gateway. Trust is currently at zero.
>
> **What they asked for:** "Can you add some validation?"
>
> **What they actually need:** a quality gate between the raw `trips` load and everything
> downstream (warehouse → BI → the surge-pricing model the ML team starts building next month),
> plus a written definition of "good data" that their own two data engineers can maintain
> after you're gone.
>
> **Constraints:** No data dictionary. The original pipeline author has left. The two remaining
> engineers are competent but have never written a data test. You have four weeks and you are
> not coming back.

Everything in this notebook is built against that brief. Keep it in view.

### The economics you'll quote in the kickoff

| Stage | Cost to fix |
|---|---|
| Prevented at entry | **$1** |
| Caught later in the pipeline | **$10** |
| After a bad decision is made | **$100** |

MetroRide paid the $100. Your job is to move them to the $1 column. That framing — not the
library — is what gets you budget for the work.

---
# Section 0 · Setup

One install, one import, one version check.

In [1]:
# Install GX Core 1.x. Takes ~60-90s on a cold Colab runtime.
%pip install -q "great_expectations>=1.0,<2.0"

# If the import in the next cell fails, do: Runtime -> Restart session, then re-run from here.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 2.0 MB/s eta 0:00:00


In [2]:
import warnings, json, os, shutil, zipfile
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import great_expectations as gx
from great_expectations.data_context.types.base import ProgressBarsConfig

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

print("great_expectations :", gx.__version__)
print("pandas             :", pd.__version__)
print("numpy              :", np.__version__)

great_expectations : 1.22.0
pandas             : 2.2.3
numpy              : 2.1.3


---
# Section 1 · The customer's data

In a real engagement you'd point GX at MetroRide's Postgres. Here we synthesise a
`trips` table that behaves the way real operational data behaves — mostly clean, with a
thin seam of genuine defects and one *legitimate* value that looks like a defect.

That last part matters more than anything else in this notebook. Hold on to it.

In [3]:
RNG = np.random.default_rng(20260420)

def make_trips(n=5_000, day="2026-04-01", seed_defects=True, drift=False):
    '''Generate one day's worth of MetroRide trip records.

    drift=True simulates what an upstream team ships without telling you:
    a renamed column and a changed payment vocabulary.
    '''
    rng = np.random.default_rng(abs(hash(day)) % (2**32))
    base = pd.Timestamp(day)

    pickup = [base + pd.Timedelta(seconds=int(s)) for s in rng.integers(0, 86_400, n)]
    duration_min = rng.gamma(shape=2.2, scale=7.0, size=n) + 2
    dropoff = [p + pd.Timedelta(minutes=float(m)) for p, m in zip(pickup, duration_min)]

    # 4-seat sedans dominate; a real minority of 6-seat MPVs exists in the fleet.
    passenger_count = rng.choice([1, 2, 3, 4, 6], size=n, p=[0.58, 0.22, 0.09, 0.08, 0.03])

    distance_km = np.round(duration_min * rng.uniform(0.25, 0.65, n), 2)
    fare_amount = np.round(45 + distance_km * 14.5 + rng.normal(0, 12, n), 2)

    df = pd.DataFrame({
        "trip_id":          [f"TRP{day.replace('-','')}{i:06d}" for i in range(n)],
        "vendor_id":        rng.choice(["MR_NORTH", "MR_SOUTH", "MR_WEST"], n),
        "pickup_datetime":  pickup,
        "dropoff_datetime": dropoff,
        "passenger_count":  passenger_count,
        "trip_distance_km": distance_km,
        "fare_amount":      fare_amount,
        "payment_type":     rng.choice(["card", "cash", "wallet"], n, p=[0.55, 0.25, 0.20]),
        "city":             "Chennai",
    })

    if seed_defects:
        # --- Defects that are genuinely wrong ---------------------------------
        idx = rng.choice(n, size=40, replace=False)
        df.loc[idx[:12],   "fare_amount"]      = -df.loc[idx[:12], "fare_amount"].abs()   # negative fares
        df.loc[idx[12:22], "fare_amount"]      = np.nan                                    # missing fares
        df.loc[idx[22:30], "dropoff_datetime"] = df.loc[idx[22:30], "pickup_datetime"] - pd.Timedelta(minutes=4)
        df.loc[idx[30:36], "trip_distance_km"] = 0.0                                       # zero-distance trips
        dupes = df.iloc[idx[36:40]].copy()                                                 # duplicate trip_ids
        df = pd.concat([df, dupes], ignore_index=True)

    if drift:
        # --- What an upstream team ships on a Tuesday without telling you -----
        df = df.rename(columns={"fare_amount": "total_fare"})       # schema drift
        df["payment_type"] = df["payment_type"].replace({"wallet": "UPI"})  # vocabulary drift

    return df.sample(frac=1, random_state=7).reset_index(drop=True)


trips_day1 = make_trips(n=5_000, day="2026-04-01")
print(f"Loaded {len(trips_day1):,} rows × {trips_day1.shape[1]} columns")
trips_day1.head()

Loaded 5,004 rows × 9 columns


,trip_id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance_km,fare_amount,payment_type,city
0,TRP20260401001745,MR_NORTH,2026-04-01 05:37:03,2026-04-01 05:51:16.842445021,1,7.47,151.00,card,Chennai
1,TRP20260401003939,MR_WEST,2026-04-01 05:33:59,2026-04-01 05:53:12.893023525,1,11.83,207.56,wallet,Chennai
2,TRP20260401002631,MR_SOUTH,2026-04-01 15:24:00,2026-04-01 15:33:22.314555647,2,5.85,128.06,card,Chennai
3,TRP20260401004306,MR_WEST,2026-04-01 07:17:52,2026-04-01 07:41:45.792841417,1,11.49,197.06,wallet,Chennai
4,TRP20260401001209,MR_NORTH,2026-04-01 12:14:11,2026-04-01 12:20:59.020235490,1,4.18,97.52,wallet,Chennai


---
# Section 2 · Discover before you enforce

**You cannot write a rule for data you have not looked at.** The single most common FDE
failure mode with GX is opening the Expectations gallery and writing rules that sound
sensible — `fare > 0`, `passenger_count <= 4` — without ever checking whether the customer's
data agrees.

Profiling answers *"what's in my data?"*. GX answers *"is my data still correct?"*.
You need the first before you can write the second.

In [4]:
def quick_profile(df: pd.DataFrame) -> pd.DataFrame:
    '''A 20-line stand-in for a profiling tool. Enough to generate candidate rules.'''
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({
            "column":     col,
            "dtype":      str(s.dtype),
            "nulls":      int(s.isna().sum()),
            "null_%":     round(100 * s.isna().mean(), 2),
            "distinct":   int(s.nunique(dropna=True)),
            "min":        s.min() if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s) else None,
            "max":        s.max() if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s) else None,
            "sample":     s.dropna().unique()[:3].tolist() if s.nunique() < 30 else None,
        })
    return pd.DataFrame(rows)

profile = quick_profile(trips_day1)
profile

,column,dtype,nulls,null_%,distinct,min,max,sample
0,trip_id,object,0,0.0,5000,None,None,None
1,vendor_id,object,0,0.0,3,None,None,"[MR_NORTH, MR_WEST, MR_SOUTH]"
2,pickup_datetime,datetime64[ns],0,0.0,4884,2026-04-01 00:00:09,2026-04-01 23:59:49,None
3,dropoff_datetime,datetime64[ns],0,0.0,5000,2026-04-01 00:06:33.317308060,2026-04-02 00:49:10.932410833,None
4,passenger_count,int64,0,0.0,5,1,6,"[1, 2, 3]"
5,trip_distance_km,float64,0,0.0,1675,0.0,44.4,None
6,fare_amount,float64,10,0.2,4459,-416.13,694.24,None
7,payment_type,object,0,0.0,3,None,None,"[card, wallet, cash]"
8,city,object,0,0.0,1,None,None,[Chennai]


In [ ]:
# Two questions the profile above should make you ask out loud:
print("Duplicate trip_ids      :", int(trips_day1["trip_id"].duplicated().sum()))
print("Negative fares          :", int((trips_day1["fare_amount"] < 0).sum()))
print("Dropoff before pickup   :", int((trips_day1["dropoff_datetime"] <= trips_day1["pickup_datetime"]).sum()))
print("Zero-distance trips     :", int((trips_day1["trip_distance_km"] == 0).sum()))
print()
print("passenger_count distribution:")
print(trips_day1["passenger_count"].value_counts().sort_index().to_string())

### 🧭 FDE Lens — profiling output is not a rule set, it's a question list

Look at the `passenger_count` distribution. There are 6-passenger trips. A junior engineer
sees that and writes `expect passenger_count <= 4` because "cars have four seats." An FDE
sees it and books fifteen minutes with the fleet ops lead.

Every anomaly in a profile has exactly two explanations:

1. **The data is wrong.** → Write the rule. Route the fix upstream.
2. **Your mental model is wrong.** → Update your model. *Then* write the rule.

You cannot tell which from the data alone. Only the customer's domain expert can, and
extracting that from them is the actual deliverable. The profile is your interview script.

### Optional: `ydata-profiling`

One line gives you a full interactive HTML EDA report. Heavy install (~2 min in Colab), so
it's commented out — uncomment if you want to see it. **Everything below works without it.**

In [6]:
 %pip install -q ydata-profiling
 from ydata_profiling import ProfileReport
 ProfileReport(trips_day1, title="MetroRide trips — Day 1", minimal=True).to_notebook_iframe()

# Note: ydata-profiling once exported a GX suite directly via .to_expectation_suite().
# That bridge is deprecated in current versions. Translate insights into Expectations yourself
# -- which is fine, because the translation step is where the domain conversation happens.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.5 MB/s eta 0:00:00


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 9/9 [00:00<00:00, 21.77it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

### 🎯 Your Turn #1 — write the interview script

Before you touch GX, write down **five candidate rules** from the profile above, and for each one
name **who at MetroRide you'd have to ask** to confirm it. Edit the cell below.

> This is not busywork. In a real engagement this list *is* your first stakeholder meeting agenda.

---
# Section 3 · The GX Core loop

Six steps. The first four you set up **once**; the last two you iterate on forever.

```
1. Data Context   ──┐
2. Data Source      │  set up once
3. Data Asset       │
4. Batch Definition ──┘
5. Expectation    ──┐  iterate every run
6. Validate       ──┘
```

---
## Step 1 of 5 — Create a Data Context

The **Data Context** is your GX project: the single entry point that manages configuration,
sources, suites, checkpoints and results.

| Mode | Where config lives | Use it for |
|---|---|---|
| `ephemeral` | In memory, vanishes on exit | Notebooks, experiments, CI runs |
| `file` | A `gx/` folder on disk | Real projects — version this in git |
| `cloud` | GX Cloud | Managed/hosted, non-coder collaboration |

The API is identical across all three. We'll use **`file`** so that Data Docs render and so
you can see the artifact you'd actually hand over.

In [7]:
PROJECT_ROOT = "/content/metroride_gx" if os.path.isdir("/content") else "./metroride_gx"

shutil.rmtree(PROJECT_ROOT, ignore_errors=True)   # clean slate so this notebook is re-runnable
os.makedirs(PROJECT_ROOT, exist_ok=True)

context = gx.get_context(mode="file", project_root_dir=PROJECT_ROOT)
context.variables.progress_bars = ProgressBarsConfig(globally=False)   # quieter output

print("Context type :", type(context).__name__)
print("Project root :", PROJECT_ROOT)
print("\nWhat GX just created on disk:")
for root, dirs, files in os.walk(os.path.join(PROJECT_ROOT, "gx")):
    depth = root.replace(PROJECT_ROOT, "").count(os.sep)
    if depth > 3: continue
    print("  " * depth + "└─ " + os.path.basename(root) + "/")

Context type : FileDataContext
Project root : /content/metroride_gx

What GX just created on disk:
  └─ gx/
    └─ expectations/
    └─ plugins/
      └─ custom_data_docs/
    └─ checkpoints/
    └─ uncommitted/
      └─ validations/
      └─ data_docs/
    └─ validation_definitions/


### 🧭 FDE Lens — the `gx/` folder is the deliverable

That directory is what you commit to the customer's repo. It contains the expectation suites,
the checkpoint config and the validation history. When you leave, this folder is what remains:
a machine-readable, reviewable, version-controlled statement of what "good data" means at
MetroRide. Treat it like production code, because it is.

---
## Step 2 of 5 — Connect a Data Source and Data Asset

- **Data Source** = *where* the data lives (a connection).
- **Data Asset**  = *which* table/file/dataframe inside it you care about.

One Source exposes many Assets. We use a pandas source because our data is already a
DataFrame in memory — that's the right choice in a notebook and in most orchestrator tasks
(Airflow/Dagster/Prefect) where a DataFrame is already in hand.

In [8]:
source = context.data_sources.add_or_update_pandas(name="metroride_lake")
asset  = source.add_dataframe_asset(name="trips")

print("Data Source :", source.name)
print("Data Asset  :", asset.name)

Data Source : metroride_lake
Data Asset  : trips


### In production you'd swap two lines

GX supports pandas, Spark, PostgreSQL, Snowflake, Databricks, BigQuery, Redshift, SQL Server,
SQLite, S3/GCS/ABS filesystems and more. **The rest of the notebook does not change.**

```python
# Postgres — validate the warehouse table in place, no data movement
source = context.data_sources.add_postgres("metroride_dw", connection_string=CONN)
asset  = source.add_table_asset(name="trips", table_name="raw_trips")

# Snowflake
source = context.data_sources.add_snowflake("mr_sf", connection_string=SF_CONN)
asset  = source.add_table_asset(name="trips", table_name="RAW_TRIPS")

# Spark, for volumes that won't fit in pandas
source = context.data_sources.add_spark("mr_spark")
asset  = source.add_dataframe_asset(name="trips")
```

**🧭 FDE Lens — push the check as close to the data as you can.** Validating a table
*in the warehouse* means GX issues SQL and never pulls rows across the network. Validating a
DataFrame means you've already paid the extraction cost. On a 400M-row table that difference
is the whole feasibility of the design. This is an architecture decision you make in week one,
and customers rarely have the vocabulary to ask you for it.

---
## Step 3 of 5 — Define the Batch

A **Batch Definition** declares *which slice* of the asset gets validated on each run.
Whole table is the simplest start; production pipelines usually validate the increment
(today's partition) so a green run means "today's load is good," not "the whole history
is still good."

In [9]:
batch_definition = asset.add_batch_definition_whole_dataframe("daily_load")
print("Batch Definition :", batch_definition.name)

# For a partitioned table you'd instead write something like:
#   asset.add_batch_definition_daily(name="by_day", column="pickup_datetime")
# so each run validates only that day's rows.

Batch Definition : daily_load


---
## Step 4 of 5 — Write your first Expectation

An **Expectation** is a single verifiable assertion about data. It reads almost like English —
that's deliberate, because a non-engineer at the customer has to be able to review it.

We'll start with the rule the junior engineer would write:

> *"passenger_count should never exceed 4 — a taxi seats four people."*

In [10]:
expectation = gx.expectations.ExpectColumnMaxToBeBetween(
    column="passenger_count",
    min_value=0,
    max_value=4,          # cars seat four... right?
)

print(expectation)

id=None meta=None notes=None result_format=<ResultFormat.BASIC: 'BASIC'> description=None catch_exceptions=False rendered_content=None severity=<FailureSeverity.CRITICAL: 'critical'> windows=None batch_id=None column='passenger_count' row_condition=None condition_parser=None min_value=0.0 max_value=4.0 strict_min=False strict_max=False


---
## Step 5 of 5 — Validate, and read the result properly

Get a batch, run the expectation, read the outcome. Note that `batch_parameters` is how the
DataFrame gets handed to GX at *runtime* — the batch definition is static config, the data is not.

In [11]:
batch  = batch_definition.get_batch(batch_parameters={"dataframe": trips_day1})
result = batch.validate(expectation)

print("success        :", result.success)
print("observed value :", result.result.get("observed_value"))

success        : False
observed value : 6


## 🔴 It failed. Now what?

`success = False`, `observed_value = 6`.

Stop and pick one of the two explanations from Section 2:

1. The data is wrong — there are corrupt 6-passenger records.
2. **Your rule is wrong** — MetroRide runs 6-seat MPVs and nobody told you.

You confirm with fleet ops. It's #2. The fleet has 6-seat vehicles.

**A few lines of code just found a real gap in your understanding of the customer's business.**
That is what data quality work actually feels like. Fix the rule, re-validate.

In [12]:
expectation.max_value = 6
result = batch.validate(expectation)

print("success        :", result.success)
print("observed value :", result.result.get("observed_value"))

success        : True
observed value : 6


### 🧭 FDE Lens — a failing expectation is a conversation, not a ticket

This is the single most important habit to build, and it's the one that separates an FDE from
a contractor who ships code.

When a new expectation fails on historical data, the default assumption is **your rule is wrong**,
not the customer's data. You've been on site three days; their fleet has existed for six years.
Every early failure is a free, high-signal invitation to a conversation with someone who knows
something you don't.

Practical consequences:

- **Never** write a suite in isolation and hand it over. It will be wrong and it will burn your credibility on day one.
- Run every draft rule against **90 days of history** before you enable it. Rules that pass 100% of history are candidates; rules that fail are questions.
- Keep a running log of "rule failed → who I asked → what I learned." That log becomes the *rationale* column of your handoff document, and it is worth more to the customer than the code.

### 🎯 Your Turn #2 — write and validate two expectations

Using the same `batch` object, write and run:

1. `fare_amount` values are between `0` and `5000`
2. `payment_type` is in the set `{card, cash, wallet}`

Useful names: `ExpectColumnValuesToBeBetween`, `ExpectColumnValuesToBeInSet`.
The full catalogue: <https://greatexpectations.io/expectations/>

For each, inspect `result.result` — how many rows were unexpected, and what were they?

In [13]:
# 1. Validate fare_amount is between 0 and 5000
exp_fare = gx.expectations.ExpectColumnValuesToBeBetween(
    column="fare_amount",
    min_value=0,
    max_value=5000
)
res_fare = batch.validate(exp_fare)
print("Fare in [0, 5000]      :", res_fare.success, "| unexpected count:", res_fare.result.get("unexpected_count"))
print("  Sample bad values    :", res_fare.result.get("partial_unexpected_list", [])[:5])
print("-" * 50)

# 2. Validate payment_type is in the expected set
exp_pay = gx.expectations.ExpectColumnValuesToBeInSet(
    column="payment_type",
    value_set=["card", "cash", "wallet"]
)
res_pay = batch.validate(exp_pay)
print("Payment type in set    :", res_pay.success, "| unexpected count:", res_pay.result.get("unexpected_count"))

Fare in [0, 5000]      : False | unexpected count: 12
  Sample bad values    : [-263.49, -222.78, -139.64, -137.87, -416.13]
--------------------------------------------------
Payment type in set    : True | unexpected count: 0


---
# Section 4 · From one rule to a data contract

One expectation is a demo. A production gate needs a bundle — an **Expectation Suite** — that
covers the dimensions of quality that matter for this table.

### The six dimensions, mapped to GX

| Dimension | Question it answers | Typical Expectation |
|---|---|---|
| **Validity** | Do values follow the rules — type, range, format? | `ExpectColumnValuesToBeBetween`, `ExpectColumnValuesToMatchRegex` |
| **Completeness** | Is anything important missing? | `ExpectColumnValuesToNotBeNull` |
| **Accuracy** | Do values reflect the real world? | `ExpectColumnValuesToBeInSet`, range checks agreed with domain experts |
| **Consistency** | Does the data agree with itself? | `ExpectColumnPairValuesAToBeGreaterThanB` |
| **Uniqueness** | Any duplicate keys? | `ExpectColumnValuesToBeUnique` |
| **Timeliness** | Is it fresh, and did it all arrive? | `ExpectTableRowCountToBeBetween`, max-timestamp checks |

Plus one that isn't a dimension but saves you more incidents than any of them:
**schema stability** — `ExpectTableColumnsToMatchSet`.

Notice how each expectation carries a `meta` block below. GX ignores it; humans don't. It's
where the *rationale* and the *owner* live, and it is the difference between a suite the
customer maintains and a suite they delete in month three because nobody knows why a rule exists.

In [14]:
suite = context.suites.add_or_update(
    gx.ExpectationSuite(name="metroride_trips_contract_v1")
)

def add(exp, why, owner, severity="blocking"):
    '''Attach rationale + ownership metadata, then register the expectation.'''
    exp.meta = {"rationale": why, "owner": owner, "severity": severity}
    suite.add_expectation(exp)

E = gx.expectations

# --- Schema stability --------------------------------------------------------
add(E.ExpectTableColumnsToMatchSet(
        column_set=list(trips_day1.columns), exact_match=True),
    why="Upstream renamed a column in Feb-2026 and silently broke revenue reporting for 11 days.",
    owner="platform-data@metroride", severity="blocking")

# --- Uniqueness --------------------------------------------------------------
add(E.ExpectColumnValuesToBeUnique(column="trip_id"),
    why="trip_id is the natural key; duplicates double-count revenue.",
    owner="platform-data@metroride", severity="blocking")

# --- Completeness ------------------------------------------------------------
add(E.ExpectColumnValuesToNotBeNull(column="trip_id"),
    why="A trip without an ID cannot be reconciled against the payment gateway.",
    owner="platform-data@metroride", severity="blocking")

add(E.ExpectColumnValuesToNotBeNull(column="fare_amount", mostly=0.999),
    why="Finance tolerates <0.1% missing fares from gateway lag; more indicates a real fault.",
    owner="finance-ops@metroride", severity="blocking")

# --- Validity ----------------------------------------------------------------
add(E.ExpectColumnValuesToMatchRegex(column="trip_id", regex=r"^TRP\d{14}$"),
    why="Format agreed with the mobile team; a change means an app-side release we weren't told about.",
    owner="mobile-platform@metroride", severity="warning")

add(E.ExpectColumnValuesToBeBetween(column="fare_amount", min_value=0, max_value=5000),
    why="Negative fares are refunds miscoded as trips. Ceiling set from 99.99pct of 2025 history.",
    owner="finance-ops@metroride", severity="blocking")

add(E.ExpectColumnValuesToBeBetween(column="passenger_count", min_value=1, max_value=6),
    why="Fleet includes 6-seat MPVs (confirmed with fleet ops, 2026-04-03). NOT 4.",
    owner="fleet-ops@metroride", severity="blocking")

add(E.ExpectColumnValuesToBeBetween(column="trip_distance_km", min_value=0.1, max_value=200),
    why="Zero-distance trips are cancelled rides leaking into the completed table.",
    owner="platform-data@metroride", severity="warning")

# --- Accuracy / controlled vocabulary ---------------------------------------
add(E.ExpectColumnValuesToBeInSet(column="payment_type", value_set=["card", "cash", "wallet"]),
    why="Closed vocabulary. A new value means an unannounced payment integration.",
    owner="payments@metroride", severity="blocking")

add(E.ExpectColumnValuesToBeInSet(column="vendor_id", value_set=["MR_NORTH", "MR_SOUTH", "MR_WEST"]),
    why="Three operating zones as of Apr-2026. A fourth means an expansion nobody told data about.",
    owner="fleet-ops@metroride", severity="warning")

# --- Consistency (cross-column) ---------------------------------------------
add(E.ExpectColumnPairValuesAToBeGreaterThanB(
        column_A="dropoff_datetime", column_B="pickup_datetime"),
    why="Negative-duration trips corrupt the ETA model's training set.",
    owner="platform-data@metroride", severity="blocking")

# --- Timeliness / volume -----------------------------------------------------
add(E.ExpectTableRowCountToBeBetween(min_value=3000, max_value=60000),
    why="Daily volume floor/ceiling from 2025 history. A short load means a partial extract.",
    owner="platform-data@metroride", severity="blocking")

print(f"Suite '{suite.name}' now holds {len(suite.expectations)} expectations\n")
for e in suite.expectations:
    col = e.configuration.kwargs.get("column", "-- table-level --")
    print(f"  [{e.meta['severity']:>8}] {e.expectation_type:<45} {col}")

Suite 'metroride_trips_contract_v1' now holds 12 expectations

  [blocking] expect_table_columns_to_match_set             -- table-level --
  [blocking] expect_column_values_to_be_unique             trip_id
  [blocking] expect_column_values_to_not_be_null           trip_id
  [blocking] expect_column_values_to_not_be_null           fare_amount
  [ warning] expect_column_values_to_match_regex           trip_id
  [blocking] expect_column_values_to_be_between            fare_amount
  [blocking] expect_column_values_to_be_between            passenger_count
  [ warning] expect_column_values_to_be_between            trip_distance_km
  [blocking] expect_column_values_to_be_in_set             payment_type
  [ warning] expect_column_values_to_be_in_set             vendor_id
  [blocking] expect_column_pair_values_a_to_be_greater_than_b -- table-level --
  [blocking] expect_table_row_count_to_be_between          -- table-level --


### 🧭 FDE Lens — the suite is a negotiated document, not a config file

Read the `owner` values again. Four different teams. That's not decoration — it's the answer to
the only question that matters at 3am when the gate goes red: **who fixes this?**

An expectation without a named owner is a pager alert with nowhere to go. It will be silenced
within a month. When you write a suite at a customer, you are implicitly assigning on-call
responsibility across their org chart, and you should do that **explicitly, in a room, with
those people present.** The code is the easy artefact; the agreement is the hard one.

Two more habits worth stealing:

- **Version the suite name** (`_v1`). Contracts change. You want to be able to say "the gate
  passed under v1 but fails under v2" instead of silently rewriting history.
- **Set ranges from history, not intuition.** `max_value=5000` came from the 99.99th percentile
  of 2025 fares, not from a guess. Write the derivation into `rationale` so the next engineer
  can re-derive it when the business changes.

---
# Section 5 · Validation Definition and Checkpoint

Two more objects take you from notebook to pipeline:

| Object | What it is | Mental model |
|---|---|---|
| **Validation Definition** | A Suite paired with the Batch it runs against | *what to check, on what data* |
| **Checkpoint** | Runs one or more validations and fires **actions** | *the runnable unit your orchestrator calls* |

The Checkpoint is the thing Airflow/Dagster/GitHub Actions actually invokes. Everything before
it is configuration.

We also set `result_format` with `unexpected_index_column_names` — that's what makes GX return
the **primary keys of the offending rows**, which is what you need for quarantine. Without it
you get counts, and counts don't let you fix anything.

In [15]:
from great_expectations.checkpoint import UpdateDataDocsAction

validation_definition = context.validation_definitions.add_or_update(
    gx.ValidationDefinition(
        name="trips_daily_validation",
        data=batch_definition,
        suite=suite,
    )
)

checkpoint = context.checkpoints.add_or_update(
    gx.Checkpoint(
        name="metroride_trips_gate",
        validation_definitions=[validation_definition],
        actions=[UpdateDataDocsAction(name="refresh_data_docs")],
        result_format={
            "result_format": "COMPLETE",
            "unexpected_index_column_names": ["trip_id"],   # <-- enables quarantine
        },
    )
)

print("Validation Definition :", validation_definition.name)
print("Checkpoint            :", checkpoint.name)
print("Actions               :", [a.name for a in checkpoint.actions])

Validation Definition : trips_daily_validation
Checkpoint            : metroride_trips_gate
Actions               : ['refresh_data_docs']


In [16]:
run = checkpoint.run(batch_parameters={"dataframe": trips_day1})

print("=" * 78)
print(f"CHECKPOINT RESULT : {'PASSED ✅' if run.success else 'FAILED ❌'}")
print("=" * 78)

for _, vres in run.run_results.items():
    st = vres.statistics
    print(f"\n{st['successful_expectations']}/{st['evaluated_expectations']} expectations passed "
          f"({st['success_percent']:.1f}%)\n")
    for r in vres.results:
        cfg  = r.expectation_config
        col  = cfg.kwargs.get("column", "")
        mark = "✅" if r.success else "❌"
        bad  = r.result.get("unexpected_count", "")
        pct  = r.result.get("unexpected_percent")
        detail = f"{bad} bad rows ({pct:.2f}%)" if pct is not None else ""
        print(f"  {mark} {cfg.type:<45} {col:<18} {detail}")

CHECKPOINT RESULT : FAILED ❌

7/12 expectations passed (58.3%)

  ✅ expect_table_columns_to_match_set                                
  ❌ expect_column_pair_values_a_to_be_greater_than_b                    8 bad rows (0.16%)
  ✅ expect_table_row_count_to_be_between                             
  ❌ expect_column_values_to_be_unique             trip_id            8 bad rows (0.16%)
  ✅ expect_column_values_to_not_be_null           trip_id            0 bad rows (0.00%)
  ✅ expect_column_values_to_match_regex           trip_id            0 bad rows (0.00%)
  ❌ expect_column_values_to_not_be_null           fare_amount        10 bad rows (0.20%)
  ❌ expect_column_values_to_be_between            fare_amount        12 bad rows (0.24%)
  ✅ expect_column_values_to_be_between            passenger_count    0 bad rows (0.00%)
  ❌ expect_column_values_to_be_between            trip_distance_km   6 bad rows (0.12%)
  ✅ expect_column_values_to_be_in_set             payment_type       0 bad rows (0.00%)

### Reading a failure properly

Notice that GX tells you three separate things for every failed expectation:

- **what** rule broke,
- **how many** rows broke it (`unexpected_count`) — is this a blip or a systemic fault?
- **which** rows (`unexpected_index_list`) — the actual keys, so you can quarantine and route them.

`unexpected_percent` is the field to build your judgment on. Twelve negative fares out of 5,000
is a coding bug in one edge case. Twelve *hundred* is an upstream schema change. Same rule,
same red light, completely different escalation path — and knowing which one you're looking at
before you wake anyone up is a large part of what the customer is paying you for.

In [17]:
# Pull out the offending trip_ids -- this is the quarantine list.
for _, vres in run.run_results.items():
    for r in vres.results:
        if r.success:
            continue
        idx = r.result.get("unexpected_index_list") or []
        ids = [i["trip_id"] for i in idx if isinstance(i, dict) and "trip_id" in i]
        print(f"\n❌ {r.expectation_config.type} ({r.expectation_config.kwargs.get('column','table')})")
        print(f"   unexpected: {r.result.get('unexpected_count')}  |  sample keys: {ids[:5]}")
        sample_vals = r.result.get("partial_unexpected_list", [])[:5]
        if sample_vals:
            print(f"   sample bad values: {sample_vals}")


❌ expect_column_pair_values_a_to_be_greater_than_b (table)
   unexpected: 8  |  sample keys: ['TRP20260401001538', 'TRP20260401002462', 'TRP20260401002192', 'TRP20260401004691', 'TRP20260401002490']
   sample bad values: [(np.datetime64('2026-04-01T06:12:19.000000000'), np.datetime64('2026-04-01T06:16:19.000000000')), (np.datetime64('2026-04-01T11:51:50.000000000'), np.datetime64('2026-04-01T11:55:50.000000000')), (np.datetime64('2026-04-01T22:51:50.000000000'), np.datetime64('2026-04-01T22:55:50.000000000')), (np.datetime64('2026-04-01T02:07:47.000000000'), np.datetime64('2026-04-01T02:11:47.000000000')), (np.datetime64('2026-04-01T11:59:57.000000000'), np.datetime64('2026-04-01T12:03:57.000000000'))]

❌ expect_column_values_to_be_unique (trip_id)
   unexpected: 8  |  sample keys: ['TRP20260401000765', 'TRP20260401002857', 'TRP20260401001660', 'TRP20260401004914', 'TRP20260401002857']
   sample bad values: ['TRP20260401000765', 'TRP20260401002857', 'TRP20260401001660', 'TRP2026040100

---
# Section 6 · Data Docs — the artefact non-engineers actually read

Every validation can render **Data Docs**: static HTML showing what you expect of the data and
how each run performed. This is the "collaborate & document" half of GX that pure validation
libraries skip, and at a customer it is frequently the highest-leverage thing you produce.

Why it matters on an engagement:

- **One source of truth** — everyone sees the same definition of "good data."
- **Faster onboarding** — a new engineer learns the dataset by reading its expectations.
- **Trust with stakeholders** — show, don't tell. Finance can see the gate ran and passed.
- **Tribal knowledge captured** — assumptions live in a reviewable artefact, not in someone's head.

In [18]:
context.build_data_docs()
docs_url = context.get_docs_sites_urls()[0]["site_url"]
docs_dir = docs_url.replace("file://", "").replace("/index.html", "")
print("Data Docs built at:", docs_dir)

# Colab can't open file:// links, so zip it up for download.
zip_path = "/content/metroride_data_docs.zip" if os.path.isdir("/content") else "./metroride_data_docs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(docs_dir):
        for f in files:
            fp = os.path.join(root, f)
            z.write(fp, os.path.relpath(fp, docs_dir))
print("Zipped to:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)      # unzip locally and open index.html
except Exception:
    print("(Not on Colab — open index.html from the path above.)")

Data Docs built at: /content/metroride_gx/gx/uncommitted/data_docs/local_site
Zipped to: /content/metroride_data_docs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**🧭 FDE Lens — where the docs live is an architecture decision.** In production you point the
Data Docs store at **S3, GCS or Azure Blob** and serve it as a static site behind the customer's
SSO. It becomes an internal URL that finance, analytics and the ML team all bookmark. That single
link does more for adoption than any amount of Slack evangelism, because it changes data quality
from a thing engineering claims to do into a thing anyone can go look at.

---
# Section 7 · The gate: pass → promote, fail → quarantine + alert

This is where GX stops being a library and becomes an **architecture component**.

```
   Source data                GX Checkpoint                 Downstream
  APIs · files · DB   ───▶   runs the suite     ──PASS──▶   Warehouse · BI · ML
                                   │
                                   └──FAIL──▶  stop the load
                                               quarantine the bad rows
                                               alert the named owner
                                               log to Data Docs
```

The function below is roughly what you'd deploy — an orchestrator-callable gate that returns a
decision, a clean frame, a quarantine frame and an alert payload.

In [19]:
def run_quality_gate(df: pd.DataFrame, cp, label: str = "batch"):
    '''Validate a batch and return a promote/block decision plus quarantined rows.

    Returns dict with: passed, clean_df, quarantine_df, failures, alerts, errored
    '''
    run = cp.run(batch_parameters={"dataframe": df})

    failures, errored, bad_keys = [], [], set()

    for _, vres in run.run_results.items():
        for r in vres.results:
            cfg  = r.expectation_config
            meta = cfg.meta or {}

            # An expectation can FAIL (data broke the rule) or ERROR (rule couldn't run).
            if r.exception_info and any(
                info.get("raised_exception") for info in r.exception_info.values()
                if isinstance(info, dict)
            ):
                errored.append({
                    "rule": cfg.type,
                    "column": cfg.kwargs.get("column"),
                    "message": next(iter(r.exception_info.values())).get("exception_message"),
                })
                continue

            if r.success:
                continue

            col = (cfg.kwargs.get("column")
                   or (f"{cfg.kwargs.get('column_A')}/{cfg.kwargs.get('column_B')}"
                       if cfg.kwargs.get("column_A") else None)
                   or cfg.kwargs.get("column_list")
                   or "<table>")

            failures.append({
                "rule":     cfg.type,
                "column":   col,
                "severity": meta.get("severity", "blocking"),
                "owner":    meta.get("owner", "unassigned"),
                "rationale": meta.get("rationale", ""),
                "bad_rows": r.result.get("unexpected_count"),
                "bad_pct":  r.result.get("unexpected_percent"),
            })
            for i in (r.result.get("unexpected_index_list") or []):
                if isinstance(i, dict) and "trip_id" in i:
                    bad_keys.add(i["trip_id"])

    blocking = [f for f in failures if f["severity"] == "blocking"]
    passed   = not blocking and not errored

    if "trip_id" in df.columns:
        quarantine_df = df[df["trip_id"].isin(bad_keys)]
        clean_df      = df[~df["trip_id"].isin(bad_keys)]
    else:
        quarantine_df, clean_df = df.iloc[0:0], df   # schema drift: no key column

    # Alert payload -- in production this POSTs to Slack / PagerDuty / email.
    # Errored expectations route to the platform team: the contract broke, not the values.
    alerts = {}
    for f in blocking:
        alerts.setdefault(f["owner"], []).append(f)
    for e in errored:
        alerts.setdefault("platform-data@metroride (schema)", []).append(e)

    return {
        "label": label, "passed": passed, "run": run,
        "clean_df": clean_df, "quarantine_df": quarantine_df,
        "failures": failures, "errored": errored, "alerts": alerts,
    }


def print_gate(g):
    print("=" * 78)
    print(f"QUALITY GATE · {g['label']} · {'✅ PROMOTE' if g['passed'] else '⛔ BLOCK'}")
    print("=" * 78)
    if g["errored"]:
        print("\n🔥 ERRORED (rule could not run at all -- usually schema drift):")
        for e in g["errored"]:
            print(f"   {e['rule']} [{e['column']}] → {e['message']}")
    if g["failures"]:
        print("\n❌ FAILED:")
        for f in g["failures"]:
            if f["bad_rows"] is None:
                detail = "table-level"
            else:
                pct = f" ({f['bad_pct']:.2f}%)" if f["bad_pct"] is not None else ""
                detail = f"{f['bad_rows']} rows{pct}"
            print(f"   [{f['severity']:>8}] {f['rule']:<45} {str(f['column']):<18} "
                  f"{detail:<20} → {f['owner']}")
    print(f"\n   promoted   : {len(g['clean_df']):,} rows")
    print(f"   quarantined: {len(g['quarantine_df']):,} rows")
    if g["alerts"]:
        print("\n📟 Alerts to send:")
        for owner, items in g["alerts"].items():
            print(f"   → {owner}: {len(items)} blocking issue(s)")

In [21]:
gate_day1 = run_quality_gate(trips_day1, checkpoint, label="2026-04-01 daily load")
print_gate(gate_day1)

QUALITY GATE · 2026-04-01 daily load · ⛔ BLOCK

❌ FAILED:
   [blocking] expect_column_pair_values_a_to_be_greater_than_b dropoff_datetime/pickup_datetime 8 rows (0.16%)       → platform-data@metroride
   [blocking] expect_column_values_to_be_unique             trip_id            8 rows (0.16%)       → platform-data@metroride
   [blocking] expect_column_values_to_not_be_null           fare_amount        10 rows (0.20%)      → finance-ops@metroride
   [blocking] expect_column_values_to_be_between            fare_amount        12 rows (0.24%)      → finance-ops@metroride
   [ warning] expect_column_values_to_be_between            trip_distance_km   6 rows (0.12%)       → platform-data@metroride

   promoted   : 4,960 rows
   quarantined: 44 rows

📟 Alerts to send:
   → platform-data@metroride: 2 blocking issue(s)
   → finance-ops@metroride: 2 blocking issue(s)


In [22]:
# What actually got quarantined? This frame is what you write to the
# quarantine table / S3 prefix for the owning team to triage.
gate_day1["quarantine_df"][
    ["trip_id", "passenger_count", "trip_distance_km", "fare_amount",
     "pickup_datetime", "dropoff_datetime", "payment_type"]
].head(10)

,trip_id,passenger_count,trip_distance_km,fare_amount,pickup_datetime,dropoff_datetime,payment_type
181,TRP20260401003361,1,1.15,NaN,2026-04-01 04:34:51,2026-04-01 04:37:29.314775325,cash
606,TRP20260401000831,6,15.67,-263.49,2026-04-01 05:13:57,2026-04-01 05:59:38.410012615,card
858,TRP20260401003362,1,0.00,133.36,2026-04-01 09:53:11,2026-04-01 10:13:12.274074087,card
871,TRP20260401000765,2,6.60,120.52,2026-04-01 22:52:42,2026-04-01 23:09:17.110329033,card
957,TRP20260401003944,1,0.00,88.61,2026-04-01 13:29:55,2026-04-01 13:40:37.273521169,cash
966,TRP20260401001538,2,10.22,180.83,2026-04-01 06:16:19,2026-04-01 06:12:19.000000000,cash
1076,TRP20260401003604,1,0.00,54.05,2026-04-01 13:46:16,2026-04-01 13:49:30.724129550,wallet
1123,TRP20260401000701,1,10.73,NaN,2026-04-01 14:45:58,2026-04-01 15:10:36.978422493,card
1164,TRP20260401004151,1,0.00,154.27,2026-04-01 13:23:23,2026-04-01 13:44:31.899350519,card
1181,TRP20260401002462,1,15.61,257.21,2026-04-01 11:55:50,2026-04-01 11:51:50.000000000,card


### 🧭 FDE Lens — "block the load" is a business decision, not a technical one

The `passed` flag above stops the pipeline. Before you ship that behaviour, someone at the
customer with authority has to answer: **is stale data worse than wrong data?**

- **Finance / regulatory reporting** → wrong is worse. Block hard. Yesterday's number is fine.
- **A real-time ops dashboard** → stale is worse. Let it through, flag it loudly, quarantine the bad slice.
- **ML training data** → wrong is much worse; the errors get baked into weights and surface months later.

Same library, opposite configuration. Get this decision in writing from a named person before
your first hard block, because the first time a gate stops a load at month-end close you want
to be pointing at an agreed policy, not defending a design choice you made alone.

**The related trap:** a gate that blocks too often gets disabled. A gate that never blocks is
decorative. You are calibrating a smoke detector — one that shrieks at toast gets its battery
pulled, and then it isn't there for the fire.

---
# Section 8 · Day 2 — the regression you were hired to catch

Overnight, an upstream team ships a change. They didn't tell data engineering, because from
their side it was a harmless refactor:

- `fare_amount` → renamed to `total_fare`
- `payment_type` → `"wallet"` becomes `"UPI"` after a payments integration

**This is exactly the class of change that cost MetroRide eleven days of wrong dashboards.**
Watch what the gate does with it.

In [23]:
trips_day2 = make_trips(n=5_000, day="2026-04-02", drift=True)
print("Day-2 columns:", list(trips_day2.columns))
print("\nDay-2 payment_type values:", sorted(trips_day2["payment_type"].unique()))
trips_day2.head(3)

Day-2 columns: ['trip_id', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance_km', 'total_fare', 'payment_type', 'city']

Day-2 payment_type values: ['UPI', 'card', 'cash']


,trip_id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance_km,total_fare,payment_type,city
0,TRP20260402001745,MR_SOUTH,2026-04-02 22:54:31,2026-04-02 23:09:06.034470375,4,6.42,141.20,card,Chennai
1,TRP20260402003939,MR_NORTH,2026-04-02 16:38:47,2026-04-02 17:04:41.978737441,1,10.45,193.49,card,Chennai
2,TRP20260402002631,MR_NORTH,2026-04-02 06:08:18,2026-04-02 06:17:39.642173522,1,6.05,123.58,UPI,Chennai


In [24]:
gate_day2 = run_quality_gate(trips_day2, checkpoint, label="2026-04-02 daily load (drift)")
print_gate(gate_day2)

QUALITY GATE · 2026-04-02 daily load (drift) · ⛔ BLOCK

🔥 ERRORED (rule could not run at all -- usually schema drift):
   expect_column_values_to_not_be_null [fare_amount] → Error: The column "fare_amount" in BatchData does not exist.
   expect_column_values_to_be_between [fare_amount] → Error: The column "fare_amount" in BatchData does not exist.

❌ FAILED:
   [blocking] expect_table_columns_to_match_set             <table>            table-level          → platform-data@metroride
   [blocking] expect_column_pair_values_a_to_be_greater_than_b dropoff_datetime/pickup_datetime 8 rows (0.16%)       → platform-data@metroride
   [blocking] expect_column_values_to_be_unique             trip_id            8 rows (0.16%)       → platform-data@metroride
   [ warning] expect_column_values_to_be_between            trip_distance_km   6 rows (0.12%)       → platform-data@metroride
   [blocking] expect_column_values_to_be_in_set             payment_type       1003 rows (20.04%)   → payments@metrori

### 🧭 FDE Lens — FAILED vs ERRORED is the distinction juniors miss

Look at the output above. Two very different things happened:

| | Meaning | What it tells you | Who you call |
|---|---|---|---|
| **FAILED** | The rule ran; the data broke it | A *data* problem — bad values got through | The data owner |
| **ERRORED** | The rule couldn't run at all | A *structural* problem — the column isn't there | The upstream/platform team |

When `fare_amount` disappears, every expectation on that column **errors** rather than fails.
A naive gate that only counts `success == False` will report those as data failures and send
finance chasing values that no longer exist. You'll lose an afternoon and some credibility.

Two things follow:

1. **Always put a schema expectation in the suite** (`ExpectTableColumnsToMatchSet`). It's the
   only rule that names the real problem — "the contract changed" — instead of describing its
   symptoms. It's the cheapest rule in the suite and it catches the most expensive class of bug.
2. **Treat an errored expectation as its own severity class.** It almost always means an
   unannounced upstream change, and the fix is a conversation with a different team than the
   one you'd call for bad values.

And note what the gate did *not* do: it didn't guess. It didn't coerce, backfill, or silently
map `UPI` to `wallet`. It stopped and told a human. Automatic repair of drifted data is how you
get an eleven-day incident that nobody notices.

### 🎯 Your Turn #3 — negotiate the drift

Upstream confirms both changes are permanent and intentional. You now have to update the contract.

Write `metroride_trips_contract_v2` that accepts the new reality:

- schema uses `total_fare` instead of `fare_amount`
- `payment_type` vocabulary is `{card, cash, UPI}`

Then re-run the gate against `trips_day2` and confirm it goes green (bar the genuine data defects).

**Then answer, in a comment:** should you have bumped to `v2`, or edited `v1` in place? Why?

In [25]:
# 1. Create the v2 suite (we never edit v1 in place!)
suite_v2 = context.suites.add_or_update(gx.ExpectationSuite(name="metroride_trips_contract_v2"))

# Helper function to add expectations with metadata (owner, rationale, severity)
def add_exp(exp, why, owner, severity="blocking"):
    exp.meta = {"rationale": why, "owner": owner, "severity": severity}
    suite_v2.add_expectation(exp)

E = gx.expectations

# Get the new columns from the drifted Day 2 data
cols_v2 = list(trips_day2.columns)

# Add expectations for the NEW reality
add_exp(E.ExpectTableColumnsToMatchSet(column_set=cols_v2, exact_match=True),
        "Schema v2: fare_amount renamed to total_fare (upstream release approved).", "platform-data@metroride")

add_exp(E.ExpectColumnValuesToBeUnique(column="trip_id"),
        "Natural key must be unique.", "platform-data@metroride")

add_exp(E.ExpectColumnValuesToNotBeNull(column="total_fare", mostly=0.999),
        "Renamed field, same tolerance as v1.", "finance-ops@metroride")

add_exp(E.ExpectColumnValuesToBeBetween(column="total_fare", min_value=0, max_value=5000),
        "Renamed field, same bounds as v1.", "finance-ops@metroride")

add_exp(E.ExpectColumnValuesToBeInSet(column="payment_type", value_set=["card", "cash", "UPI"]),
        "Vocabulary v2: wallet -> UPI after payments integration.", "payments@metroride")

add_exp(E.ExpectColumnValuesToBeBetween(column="passenger_count", min_value=1, max_value=6),
        "Fleet includes 6-seat MPVs.", "fleet-ops@metroride")

print("✅ Contract v2 created successfully with", len(suite_v2.expectations), "expectations.")

✅ Contract v2 created successfully with 6 expectations.


---
# Section 9 · Judgment — severity tiers and alert fatigue

Every rule in your suite carries a `severity`. That single field is what keeps the gate alive
past month three.

| Tier | Behaviour | Use it when |
|---|---|---|
| **blocking** | Stop the load. Page the owner. | Wrong data would cause a wrong decision or corrupt a model |
| **warning** | Promote the data. Log it. Digest weekly. | Suspicious but tolerable; you want a trend, not a page |
| **observability** | Never alerts. Metric only. | Row counts, distributions — you're watching for drift over time |

Let's see what happens to Day 1 under a warnings-only policy.

In [26]:
def summarise_by_severity(gate):
    if not gate["failures"]:
        print("No failures."); return
    rows = pd.DataFrame(gate["failures"])
    print(rows[["severity", "rule", "column", "bad_rows", "bad_pct", "owner"]].to_string(index=False))
    print("\nBy severity:")
    print(rows.groupby("severity").size().to_string())
    blocking = (rows["severity"] == "blocking").sum()
    print(f"\nDecision: {'⛔ BLOCK' if blocking else '⚠️  PROMOTE WITH WARNINGS'}")

summarise_by_severity(gate_day1)

severity                                             rule                           column  bad_rows  bad_pct                   owner
blocking expect_column_pair_values_a_to_be_greater_than_b dropoff_datetime/pickup_datetime         8 0.159872 platform-data@metroride
blocking                expect_column_values_to_be_unique                          trip_id         8 0.159872 platform-data@metroride
blocking              expect_column_values_to_not_be_null                      fare_amount        10 0.199840   finance-ops@metroride
blocking               expect_column_values_to_be_between                      fare_amount        12 0.240288   finance-ops@metroride
 warning               expect_column_values_to_be_between                 trip_distance_km         6 0.119904 platform-data@metroride

By severity:
severity
blocking    4
warning     1

Decision: ⛔ BLOCK


### 🧭 FDE Lens — the alerting economics

A gate that fires ten times a week gets a Slack channel muted within a month, and then it is
worse than no gate at all, because the customer *believes* they're protected.

Rules of thumb from real engagements:

- **Start almost everything as `warning`.** Run for two weeks. Promote to `blocking` only the
  rules that stayed quiet — those are the ones where a red light genuinely means something.
- **Budget your pages.** If more than ~2 blocking alerts/week reach a human, the suite is
  miscalibrated, not the data. Fix the suite.
- **Use `mostly=`** for rules that are true 99.9% of the time rather than 100%. `mostly=0.999`
  on `fare_amount` not-null tolerates gateway lag without dropping the rule entirely. A rule
  that's directionally right and quiet beats a rule that's perfect and muted.
- **Every alert names an owner and a runbook line.** "fare_amount has 400 nulls" is noise.
  "fare_amount nulls exceeded 0.1% — likely gateway lag; check payments-api dashboard;
  owner finance-ops" is an action.

This calibration work is invisible in a demo and it is most of the value you deliver. Nobody
will thank you for it and it's the reason the gate is still running a year later.

---
# Section 10 · Profiling vs Validation — choosing the right tool

| | **ydata-profiling** | **Great Expectations** |
|---|---|---|
| **Purpose** | Explore & understand data | Validate & enforce data quality |
| **Answers** | "What's in my data?" | "Is my data still correct?" |
| **Output** | One-off HTML report + stats | Pass/fail results + Data Docs |
| **When you run it** | Ad-hoc, at the start of a project | Every run, in CI/CD and pipelines |
| **Pass / fail?** | No — descriptive only | Yes — assertion-based |
| **Alerting** | No | Yes, on failure |
| **Best for** | Discovery & first-look EDA | Monitoring & regression safety |

**They are not rivals. Profiling helps you discover the rules; GX enforces them.**

```
  ydata-profiling  ──▶  You (the analyst)  ──▶  Great Expectations
  profile the raw       turn insights into      enforce on every run,
  dataset               candidate rules         alert on drift
```

⚠️ ydata-profiling once auto-exported a GX suite via `to_expectation_suite()`. That bridge is
**deprecated** in current versions — translate insights into Expectations yourself.

**🧭 FDE Lens — and that deprecation is a gift.** The manual translation step is precisely where
the domain conversation happens. An auto-generated suite encodes whatever was in last Tuesday's
data, including its bugs, with no rationale and no owner. A hand-written one encodes what the
business *means*. You want the friction.

---
# Section 11 · The handoff artefact

You are not coming back. Whatever isn't written down doesn't survive.

The cell below generates a data-contract document straight from the suite metadata — rules,
rationale, owners, severities. In an engagement this goes into the customer's repo next to the
`gx/` folder and gets reviewed by every named owner **before** the gate goes live.

In [27]:
def generate_contract_doc(suite, table="metroride.trips") -> str:
    lines = [
        f"# Data Contract — `{table}`",
        f"", f"**Suite:** `{suite.name}`  ",
        f"**Expectations:** {len(suite.expectations)}  ",
        f"**Generated:** {pd.Timestamp.now():%Y-%m-%d}  ",
        f"**Prepared by:** Forward Deployed Engineering, ArchitectsVibe",
        "",
        "## How to read this",
        "",
        "Each row is an enforced rule. `blocking` rules stop the daily load and page the owner;",
        "`warning` rules let the load through and appear in the weekly digest. The owner is the",
        "team accountable for fixing violations — not the team that runs the pipeline.",
        "",
        "## Enforced rules", "",
        "| Severity | Rule | Column | Owner | Why it exists |",
        "|---|---|---|---|---|",
    ]
    for e in suite.expectations:
        m   = e.meta or {}
        col = e.configuration.kwargs.get("column", "*table*")
        lines.append(
            f"| `{m.get('severity','blocking')}` | `{e.expectation_type}` | `{col}` "
            f"| {m.get('owner','unassigned')} | {m.get('rationale','')} |"
        )
    lines += [
        "", "## Operating procedure", "",
        "1. Checkpoint `metroride_trips_gate` runs after each daily load.",
        "2. **PASS** → data promoted to the warehouse; Data Docs updated.",
        "3. **FAIL (blocking)** → load halted, offending rows written to the quarantine table,",
        "   owning team alerted. Owner triages within one business day.",
        "4. **ERRORED** → treat as an upstream contract breach, not a data defect. Escalate to",
        "   the platform team; do not attempt to repair the batch.",
        "5. Contract changes require a version bump (`_v2`) and sign-off from every affected owner.",
        "",
        "## Open questions for MetroRide", "",
        "- [ ] Confirm the fare ceiling (currently ₹5,000) against 2026 pricing.",
        "- [ ] Agree the retention policy for the quarantine table.",
        "- [ ] Name a secondary on-call for `platform-data`.",
    ]
    return "\n".join(lines)


contract_md = generate_contract_doc(suite)

out_path = "/content/metroride_data_contract.md" if os.path.isdir("/content") else "./metroride_data_contract.md"
with open(out_path, "w") as f:
    f.write(contract_md)

from IPython.display import Markdown, display
display(Markdown(contract_md))
print("\nSaved to:", out_path)

# Data Contract — `metroride.trips`

**Suite:** `metroride_trips_contract_v1`  
**Expectations:** 12  
**Generated:** 2026-09-07  
**Prepared by:** Forward Deployed Engineering, ArchitectsVibe

## How to read this

Each row is an enforced rule. `blocking` rules stop the daily load and page the owner;
`warning` rules let the load through and appear in the weekly digest. The owner is the
team accountable for fixing violations — not the team that runs the pipeline.

## Enforced rules

| Severity | Rule | Column | Owner | Why it exists |
|---|---|---|---|---|
| `blocking` | `expect_table_columns_to_match_set` | `*table*` | platform-data@metroride | Upstream renamed a column in Feb-2026 and silently broke revenue reporting for 11 days. |
| `blocking` | `expect_column_values_to_be_unique` | `trip_id` | platform-data@metroride | trip_id is the natural key; duplicates double-count revenue. |
| `blocking` | `expect_column_values_to_not_be_null` | `trip_id` | platform-data@metroride | A trip without an ID cannot be reconciled against the payment gateway. |
| `blocking` | `expect_column_values_to_not_be_null` | `fare_amount` | finance-ops@metroride | Finance tolerates <0.1% missing fares from gateway lag; more indicates a real fault. |
| `warning` | `expect_column_values_to_match_regex` | `trip_id` | mobile-platform@metroride | Format agreed with the mobile team; a change means an app-side release we weren't told about. |
| `blocking` | `expect_column_values_to_be_between` | `fare_amount` | finance-ops@metroride | Negative fares are refunds miscoded as trips. Ceiling set from 99.99pct of 2025 history. |
| `blocking` | `expect_column_values_to_be_between` | `passenger_count` | fleet-ops@metroride | Fleet includes 6-seat MPVs (confirmed with fleet ops, 2026-04-03). NOT 4. |
| `warning` | `expect_column_values_to_be_between` | `trip_distance_km` | platform-data@metroride | Zero-distance trips are cancelled rides leaking into the completed table. |
| `blocking` | `expect_column_values_to_be_in_set` | `payment_type` | payments@metroride | Closed vocabulary. A new value means an unannounced payment integration. |
| `warning` | `expect_column_values_to_be_in_set` | `vendor_id` | fleet-ops@metroride | Three operating zones as of Apr-2026. A fourth means an expansion nobody told data about. |
| `blocking` | `expect_column_pair_values_a_to_be_greater_than_b` | `*table*` | platform-data@metroride | Negative-duration trips corrupt the ETA model's training set. |
| `blocking` | `expect_table_row_count_to_be_between` | `*table*` | platform-data@metroride | Daily volume floor/ceiling from 2025 history. A short load means a partial extract. |

## Operating procedure

1. Checkpoint `metroride_trips_gate` runs after each daily load.
2. **PASS** → data promoted to the warehouse; Data Docs updated.
3. **FAIL (blocking)** → load halted, offending rows written to the quarantine table,
   owning team alerted. Owner triages within one business day.
4. **ERRORED** → treat as an upstream contract breach, not a data defect. Escalate to
   the platform team; do not attempt to repair the batch.
5. Contract changes require a version bump (`_v2`) and sign-off from every affected owner.

## Open questions for MetroRide

- [ ] Confirm the fare ceiling (currently ₹5,000) against 2026 pricing.
- [ ] Agree the retention policy for the quarantine table.
- [ ] Name a secondary on-call for `platform-data`.


Saved to: /content/metroride_data_contract.md


### The FDE handoff checklist

Before you leave the engagement, all of these are true:

- [ ] The `gx/` folder is committed to **the customer's** repo, not yours.
- [ ] The checkpoint is invoked by **their** orchestrator, on **their** schedule.
- [ ] Data Docs are published to a URL their stakeholders can reach without you.
- [ ] Every rule has a named owner who has **seen the rule and agreed to own it**.
- [ ] Two of their engineers have added a new expectation themselves, with you watching, not typing.
- [ ] The blocking-vs-warning policy is written down and signed off by someone with authority.
- [ ] There is a runbook for "the gate is red at 3am" that doesn't contain your phone number.
- [ ] You've shown them the Expectations gallery so they know how to extend the suite.

**🧭 FDE Lens.** The last two are the ones people skip and the ones that determine whether the
work survives. Delivering a working gate is table stakes. Delivering a customer team that can
*change* the gate without you is the job.

---
# 🎯 Capstone — the surge-pricing feature table

MetroRide's ML team is building surge pricing. They've asked you to gate the feature table
that trains it. From the brief:

- Features are aggregated **per zone, per hour**.
- `surge_multiplier` must sit between `1.0` and `5.0`. Above 5.0 has legal exposure — regulators
  have opened cases on ride-hailing surge caps.
- Every `(zone, hour)` combination must be **unique**.
- `demand_index` should never be null; the model can't impute it meaningfully.
- Exactly `3 zones × 24 hours = 72` rows per day. Fewer means a partial aggregation.

Build a suite, a validation definition, a checkpoint, and run the gate against `features_day1`.

**Then write two paragraphs, not code:**

1. Which rules did you make `blocking` and which `warning`? Justify each against the
   consequence of being wrong. Remember: this feeds a *model*, not a dashboard.
2. The gate goes red at 2am on a Saturday, on `surge_multiplier`. Who do you wake, and what
   do you tell them in the first sentence?

In [28]:
def make_features(day="2026-04-01", broken=False):
    rng = np.random.default_rng(abs(hash(day + "feat")) % (2**32))
    zones = ["MR_NORTH", "MR_SOUTH", "MR_WEST"]
    rows = []
    for z in zones:
        for h in range(24):
            rows.append({
                "zone": z, "hour": h, "date": day,
                "demand_index":     round(float(rng.uniform(0.2, 3.5)), 3),
                "supply_index":     round(float(rng.uniform(0.3, 2.8)), 3),
                "surge_multiplier": round(float(np.clip(rng.normal(1.4, 0.5), 1.0, 3.0)), 2),
                "trips_completed":  int(rng.integers(20, 900)),
            })
    df = pd.DataFrame(rows)
    if broken:
        df.loc[5,  "surge_multiplier"] = 7.5      # regulatory exposure
        df.loc[12, "demand_index"]     = np.nan
        df = pd.concat([df, df.iloc[[3]]], ignore_index=True)   # duplicate (zone, hour)
        df = df.drop(index=[40, 41]).reset_index(drop=True)     # partial aggregation
    return df

features_day1 = make_features("2026-04-01", broken=True)
print(features_day1.shape)
features_day1.head()

(71, 7)


,zone,hour,date,demand_index,supply_index,surge_multiplier,trips_completed
0,MR_NORTH,0,2026-04-01,1.447,0.770,1.87,475
1,MR_NORTH,1,2026-04-01,2.016,2.247,1.38,190
2,MR_NORTH,2,2026-04-01,2.668,2.488,1.51,792
3,MR_NORTH,3,2026-04-01,1.998,0.741,1.78,811
4,MR_NORTH,4,2026-04-01,2.644,2.554,1.38,475


In [29]:
# 1. Setup Source, Asset, and Batch Definition
feat_source = context.data_sources.add_or_update_pandas("metroride_ml")
feat_asset  = feat_source.add_dataframe_asset(name="surge_features")
feat_bd     = feat_asset.add_batch_definition_whole_dataframe("hourly_features")

# 2. Create the Suite
feat_suite = context.suites.add_or_update(gx.ExpectationSuite(name="surge_features_contract_v1"))

# Helper function to add expectations with metadata
def add_feat(exp, why, owner, severity="blocking"):
    exp.meta = {"rationale": why, "owner": owner, "severity": severity}
    feat_suite.add_expectation(exp)

E = gx.expectations

# Rule 1: Regulatory cap (BLOCKING - legal exposure)
add_feat(E.ExpectColumnValuesToBeBetween(column="surge_multiplier", min_value=1.0, max_value=5.0),
         "Regulatory cap. A value > 5.0 is a legal exposure, not a data bug.", "ml-platform@metroride", "blocking")

# Rule 2: Unique zone/hour combo (BLOCKING - prevents model bias)
add_feat(E.ExpectCompoundColumnsToBeUnique(column_list=["zone", "hour"]),
         "Duplicate (zone, hour) double-weights an observation and silently biases the model.", "ml-platform@metroride", "blocking")

# Rule 3: No nulls in primary feature (BLOCKING)
add_feat(E.ExpectColumnValuesToNotBeNull(column="demand_index"),
         "Primary feature; cannot be meaningfully imputed.", "ml-platform@metroride", "blocking")

# Rule 4: Exact row count (BLOCKING - catches partial aggregations)
add_feat(E.ExpectTableRowCountToEqual(value=72),
         "3 zones x 24 hours = 72. Anything else is a partial aggregation.", "platform-data@metroride", "blocking")

# 3. Create Validation Definition and Checkpoint
feat_vd = context.validation_definitions.add_or_update(
    gx.ValidationDefinition(name="surge_features_validation", data=feat_bd, suite=feat_suite))

feat_cp = context.checkpoints.add_or_update(gx.Checkpoint(
    name="surge_features_gate",
    validation_definitions=[feat_vd],
    actions=[gx.checkpoint.UpdateDataDocsAction(name="refresh_docs_feat")],
    result_format={"result_format": "COMPLETE"},
))

# 4. Run the gate against the broken Day 1 data
print("Running gate against broken data...")
feat_run = feat_cp.run(batch_parameters={"dataframe": features_day1})

print("=" * 70)
print("SURGE FEATURE GATE :", "PASSED ✅" if feat_run.success else "FAILED ❌ (As expected!)")
print("=" * 70)

# Print the results
for _, v in feat_run.run_results.items():
    for r in v.results:
        cfg = r.expectation_config
        status = "✅" if r.success else "❌"
        col = cfg.kwargs.get("column", cfg.kwargs.get("column_list", "<table>"))
        sev = (cfg.meta or {}).get("severity", "-")
        print(f"  {status} {cfg.type:<45} {str(col):<22} [{sev}]")

Running gate against broken data...
SURGE FEATURE GATE : FAILED ❌ (As expected!)
  ❌ expect_column_values_to_be_between            surge_multiplier       [blocking]
  ❌ expect_compound_columns_to_be_unique          ['zone', 'hour']       [blocking]
  ❌ expect_table_row_count_to_equal               <table>                [blocking]
  ❌ expect_column_values_to_not_be_null           demand_index           [blocking]


---
# 📖 Solutions

Try the exercises first. Genuinely — the API is not the hard part, and reading a solution to a
judgment question teaches you nothing.

In [30]:
# ---------- Your Turn #2 ----------------------------------------------------
exp_fare = gx.expectations.ExpectColumnValuesToBeBetween(
    column="fare_amount", min_value=0, max_value=5000)
res = batch.validate(exp_fare)
print("fare in [0, 5000]      :", res.success, "| unexpected:", res.result.get("unexpected_count"))
print("  sample bad values    :", res.result.get("partial_unexpected_list", [])[:5])

exp_pay = gx.expectations.ExpectColumnValuesToBeInSet(
    column="payment_type", value_set=["card", "cash", "wallet"])
res = batch.validate(exp_pay)
print("payment_type in set    :", res.success, "| unexpected:", res.result.get("unexpected_count"))

fare in [0, 5000]      : False | unexpected: 12
  sample bad values    : [-263.49, -222.78, -139.64, -137.87, -416.13]
payment_type in set    : True | unexpected: 0


In [31]:
# ---------- Your Turn #3 : contract v2 --------------------------------------
suite_v2 = context.suites.add_or_update(gx.ExpectationSuite(name="metroride_trips_contract_v2"))

def add2(exp, why, owner, severity="blocking"):
    exp.meta = {"rationale": why, "owner": owner, "severity": severity}
    suite_v2.add_expectation(exp)

E = gx.expectations
cols_v2 = list(trips_day2.columns)

add2(E.ExpectTableColumnsToMatchSet(column_set=cols_v2, exact_match=True),
     "Schema v2: fare_amount renamed to total_fare (upstream release 2026-04-02, approved).",
     "platform-data@metroride")
add2(E.ExpectColumnValuesToBeUnique(column="trip_id"),
     "Natural key.", "platform-data@metroride")
add2(E.ExpectColumnValuesToNotBeNull(column="fare_amount".replace("fare_amount", "total_fare"), mostly=0.999),
     "Renamed field, same tolerance as v1.", "finance-ops@metroride")
add2(E.ExpectColumnValuesToBeBetween(column="total_fare", min_value=0, max_value=5000),
     "Renamed field, same bounds as v1.", "finance-ops@metroride")
add2(E.ExpectColumnValuesToBeInSet(column="payment_type", value_set=["card", "cash", "UPI"]),
     "Vocabulary v2: wallet -> UPI after the payments integration (confirmed 2026-04-02).",
     "payments@metroride")
add2(E.ExpectColumnValuesToBeBetween(column="passenger_count", min_value=1, max_value=6),
     "Fleet includes 6-seat MPVs.", "fleet-ops@metroride")
add2(E.ExpectColumnPairValuesAToBeGreaterThanB(
        column_A="dropoff_datetime", column_B="pickup_datetime"),
     "Negative durations corrupt the ETA model.", "platform-data@metroride")
add2(E.ExpectTableRowCountToBeBetween(min_value=3000, max_value=60000),
     "Daily volume envelope.", "platform-data@metroride")

vd_v2 = context.validation_definitions.add_or_update(
    gx.ValidationDefinition(name="trips_daily_validation_v2", data=batch_definition, suite=suite_v2))
cp_v2 = context.checkpoints.add_or_update(gx.Checkpoint(
    name="metroride_trips_gate_v2",
    validation_definitions=[vd_v2],
    actions=[UpdateDataDocsAction(name="refresh_docs_v2")],
    result_format={"result_format": "COMPLETE", "unexpected_index_column_names": ["trip_id"]},
))

gate_v2 = run_quality_gate(trips_day2, cp_v2, label="2026-04-02 under contract v2")
print_gate(gate_v2)

QUALITY GATE · 2026-04-02 under contract v2 · ⛔ BLOCK

❌ FAILED:
   [blocking] expect_column_pair_values_a_to_be_greater_than_b dropoff_datetime/pickup_datetime 8 rows (0.16%)       → platform-data@metroride
   [blocking] expect_column_values_to_be_unique             trip_id            8 rows (0.16%)       → platform-data@metroride
   [blocking] expect_column_values_to_not_be_null           total_fare         10 rows (0.20%)      → finance-ops@metroride
   [blocking] expect_column_values_to_be_between            total_fare         12 rows (0.24%)      → finance-ops@metroride

   promoted   : 4,966 rows
   quarantined: 38 rows

📟 Alerts to send:
   → platform-data@metroride: 2 blocking issue(s)
   → finance-ops@metroride: 2 blocking issue(s)


**Why `v2` rather than editing `v1` in place**

Because the contract is evidence, not configuration.

If you edit `v1`, you destroy the ability to say *"this batch passed the contract that was in
force on 1 April."* When finance re-opens the eleven-day incident in six months and asks which
rules were live at the time, you need an answer. Version bumps also force the sign-off
conversation — a new version is a document someone has to approve, whereas an edit is something
one engineer does quietly on a Friday.

Same discipline as a database migration, and for the same reason: history has to stay readable.

In [32]:
# ---------- Capstone ---------------------------------------------------------
feat_source = context.data_sources.add_or_update_pandas("metroride_ml")
feat_asset  = feat_source.add_dataframe_asset(name="surge_features")
feat_bd     = feat_asset.add_batch_definition_whole_dataframe("hourly_features")

feat_suite = context.suites.add_or_update(gx.ExpectationSuite(name="surge_features_contract_v1"))

def addf(exp, why, owner, severity="blocking"):
    exp.meta = {"rationale": why, "owner": owner, "severity": severity}
    feat_suite.add_expectation(exp)

addf(E.ExpectColumnValuesToBeBetween(column="surge_multiplier", min_value=1.0, max_value=5.0),
     "Regulatory cap. A value above 5.0 that reaches production is a legal exposure, not a data bug.",
     "ml-platform@metroride", "blocking")
addf(E.ExpectCompoundColumnsToBeUnique(column_list=["zone", "hour"]),
     "Duplicate (zone,hour) double-weights an observation and silently biases the model.",
     "ml-platform@metroride", "blocking")
addf(E.ExpectColumnValuesToNotBeNull(column="demand_index"),
     "Primary feature; cannot be meaningfully imputed.",
     "ml-platform@metroride", "blocking")
addf(E.ExpectTableRowCountToEqual(value=72),
     "3 zones x 24 hours. Anything else is a partial aggregation.",
     "platform-data@metroride", "blocking")
addf(E.ExpectColumnValuesToBeInSet(column="zone", value_set=["MR_NORTH", "MR_SOUTH", "MR_WEST"]),
     "Three operating zones as of Apr-2026.", "fleet-ops@metroride", "warning")
addf(E.ExpectColumnValuesToBeBetween(column="trips_completed", min_value=0, max_value=5000),
     "Sanity envelope on hourly volume.", "platform-data@metroride", "warning")

feat_vd = context.validation_definitions.add_or_update(
    gx.ValidationDefinition(name="surge_features_validation", data=feat_bd, suite=feat_suite))
feat_cp = context.checkpoints.add_or_update(gx.Checkpoint(
    name="surge_features_gate",
    validation_definitions=[feat_vd],
    actions=[UpdateDataDocsAction(name="refresh_docs_feat")],
    result_format={"result_format": "COMPLETE"},
))

feat_run = feat_cp.run(batch_parameters={"dataframe": features_day1})
print("=" * 78)
print("SURGE FEATURE GATE :", "PASSED ✅" if feat_run.success else "FAILED ❌")
print("=" * 78)
for _, v in feat_run.run_results.items():
    for r in v.results:
        cfg = r.expectation_config
        print(f"  {'✅' if r.success else '❌'} {cfg.type:<45} "
              f"{str(cfg.kwargs.get('column', cfg.kwargs.get('column_list','<table>'))):<22} "
              f"[{(cfg.meta or {}).get('severity','-')}]")

SURGE FEATURE GATE : FAILED ❌
  ❌ expect_column_values_to_be_between            surge_multiplier       [blocking]
  ❌ expect_compound_columns_to_be_unique          ['zone', 'hour']       [blocking]
  ❌ expect_table_row_count_to_equal               <table>                [blocking]
  ❌ expect_column_values_to_not_be_null           demand_index           [blocking]
  ✅ expect_column_values_to_be_in_set             zone                   [warning]
  ✅ expect_column_values_to_be_between            trips_completed        [warning]


**Capstone discussion — the two paragraphs**

*On severity.* Almost everything here is `blocking`, and that's unusual — it's justified because
this table trains a model rather than fills a dashboard. Bad rows in a dashboard are visible and
correctable; bad rows in training data get compressed into weights and surface months later as
inexplicable pricing behaviour, by which point the causal chain back to one bad Saturday is gone.
The `surge_multiplier` ceiling is blocking for a different reason again: it isn't a data-quality
rule at all, it's a **compliance control** that happens to be implementable as one. When a rule
has a regulator behind it, "promote with a warning" is not an available option. `zone` membership
is only a warning because a fourth zone means the business expanded — surprising, worth a
conversation, but not a reason to stop tonight's training run.

*On the 2am page.* You wake the ML platform on-call, and the first sentence is
**"Surge feature generation produced a multiplier above the 5.0 regulatory cap; the gate blocked
promotion, so nothing reached the model — this is a triage, not an outage."** That sentence does
three things in order: names the business risk, states that the risk was contained, and sets the
urgency correctly. What you do *not* lead with is `expect_column_values_to_be_between failed on
surge_multiplier` — that's the same fact rendered useless to a person who has been awake for
forty seconds. Translating a validation result into a consequence is the last mile of this
entire discipline, and it's the part no library does for you.

---
# Wrap-up

### The four things to remember

1. **Expectations are unit tests for data.** Declarative, verifiable, readable assertions.
2. **GX Core is free and code-first.** `context → source → asset → batch → expectation → validate`.
3. **Scale with Suites and Checkpoints.** One check becomes an automated gate with alerts.
4. **Profile to discover, GX to enforce.** Profiling explains data; GX protects it.

### The four things an FDE remembers instead

1. **A failing expectation is a conversation.** Your rule is wrong more often than their data is.
2. **Every rule needs an owner.** An unowned alert is a muted channel.
3. **FAILED ≠ ERRORED.** One is a data problem, the other is a broken contract with a different team.
4. **The deliverable is a customer team that can change the gate without you.**

---

### Where to go next

| | |
|---|---|
| Docs | <https://docs.greatexpectations.io> |
| Expectations gallery | <https://greatexpectations.io/expectations/> |
| Custom expectations | For business rules the gallery doesn't cover |
| GX Cloud | Managed UI, scheduling, alerting, non-coder collaboration |

**Extend this lab yourself:**

- Add a **Slack action** to the checkpoint and watch a real alert fire.
- Point the Data Docs store at **S3** and serve it as a static site.
- Wrap `run_quality_gate` in an **Airflow / Dagster** task and fail the DAG on a blocking result.
- Write a **custom Expectation** for a rule the gallery doesn't have.
- Run the suite against **90 days of history** and count how many rules you'd have to soften.

---

<div align="center">

**TECHADEMY · FDE TRAINING**

*Questions? Let's validate some data.*

</div>